In [1]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score

In [2]:
import pandas as pd
df = pd.read_csv("lifestyle_sustainability_data.csv")
print(df.head(5))

   ParticipantID  Age  Location             DietType LocalFoodFrequency  \
0              1   35     Urban   Mostly Plant-Based              Often   
1              2   28  Suburban             Balanced          Sometimes   
2              3   65     Rural  Mostly Animal-Based             Rarely   
3              4   42     Urban   Mostly Plant-Based              Often   
4              5   31  Suburban             Balanced          Sometimes   

  TransportationMode   EnergySource   HomeType  HomeSize ClothingFrequency  \
0               Bike      Renewable  Apartment       800            Rarely   
1     Public Transit          Mixed      House      1500         Sometimes   
2                Car  Non-Renewable      House      2500             Often   
3               Walk      Renewable  Apartment       950         Sometimes   
4     Public Transit          Mixed      House      1800             Often   

   SustainableBrands  EnvironmentalAwareness CommunityInvolvement  \
0          

In [3]:
from sklearn.preprocessing import LabelEncoder

le_dict = {}

for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le

# ── Print encoding reference so you know which number = which label ──
print("Encoding reference:")
for col, le in le_dict.items():
    print(f"  {col}: { {i: label for i, label in enumerate(le.classes_)} }")

Encoding reference:
  Location: {0: 'Rural', 1: 'Suburban', 2: 'Urban'}
  DietType: {0: 'Balanced', 1: 'Mostly Animal-Based', 2: 'Mostly Plant-Based'}
  LocalFoodFrequency: {0: 'Always', 1: 'Often', 2: 'Rarely', 3: 'Sometimes'}
  TransportationMode: {0: 'Bike', 1: 'Car', 2: 'Public Transit', 3: 'Walk'}
  EnergySource: {0: 'Mixed', 1: 'Non-Renewable', 2: 'Renewable'}
  HomeType: {0: 'Apartment', 1: 'House', 2: 'Other'}
  ClothingFrequency: {0: 'Always', 1: 'Often', 2: 'Rarely', 3: 'Sometimes'}
  CommunityInvolvement: {0: 'High', 1: 'Low', 2: 'Moderate', 3: nan}
  Gender: {0: 'Female', 1: 'Male', 2: 'Non-Binary', 3: 'Prefer not to say'}
  UsingPlasticProducts: {0: 'Never', 1: 'Often', 2: 'Rarely', 3: 'Sometimes'}
  DisposalMethods: {0: 'Combination', 1: 'Composting', 2: 'Landfill', 3: 'Recycling'}
  PhysicalActivities: {0: 'High', 1: 'Low', 2: 'Moderate', 3: nan}


In [4]:
print(df.columns)

Index(['ParticipantID', 'Age', 'Location', 'DietType', 'LocalFoodFrequency',
       'TransportationMode', 'EnergySource', 'HomeType', 'HomeSize',
       'ClothingFrequency', 'SustainableBrands', 'EnvironmentalAwareness',
       'CommunityInvolvement', 'MonthlyElectricityConsumption',
       'MonthlyWaterConsumption', 'Gender', 'UsingPlasticProducts',
       'DisposalMethods', 'PhysicalActivities', 'Rating'],
      dtype='object')


In [5]:
X = df.drop(['ParticipantID', 'Rating'], axis=1)
Y = df['Rating']

In [6]:
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

# Step 1: Split FIRST
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

# Step 1: Train initial model
model = XGBRegressor()
model.fit(X_train, y_train)

# Step 5: Train improved model
model = XGBRegressor(
    n_estimators=150,
    max_depth=3,
    learning_rate=0.08,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=1,
    reg_lambda=3,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    #early_stopping_rounds=20,
    verbose=False
)

# Step 6: Predict
y_pred = model.predict(X_test)

# Step 7: Evaluate
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(((y_test - y_pred) ** 2).mean())
r2 = r2_score(y_test, y_pred)

accuracy = 100 - (mae * 20)

print("Train R2:", model.score(X_train, y_train))
print("Test R2:", model.score(X_test, y_test))

print("Approx Accuracy:", accuracy, "%")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

Train R2: 0.9234245419502258
Test R2: 0.7060117721557617
Approx Accuracy: 88.23297381401062 %
MAE: 0.588351309299469
RMSE: 0.8596842621213465
R2: 0.7060117721557617


In [7]:
# ── CELL 7: Collect user input in simple, friendly language ──

def get_user_input():
    user = {}

    print("=" * 55)
    print("   🌱 SUSTAINABILITY LIFESTYLE QUESTIONNAIRE")
    print("=" * 55)

    # ── About You ──
    print("\n--- 👤 About You ---")
    user['Age'] = int(input("How old are you? (e.g. 20): "))

    print("What is your gender?")
    print("  0 = Female   1 = Male   2 = Other / Prefer not to say")
    user['Gender'] = int(input("Enter number: "))

    print("What type of area do you live in?")
    print("  0 = Village / Small town")
    print("  1 = Outskirts of a city (like Gandhinagar near Ahmedabad)")
    print("  2 = City center (like Ahmedabad, Mumbai, Delhi)")
    user['Location'] = int(input("Enter number: "))

    # ── Diet & Food ──
    print("\n--- 🥗 Diet & Food ---")
    print("What best describes your daily diet?")
    print("  0 = Mix of everything (dal, roti, chicken, veggies etc.)")
    print("  1 = Mostly non-veg (chicken, fish, eggs, meat)")
    print("  2 = Mostly veg / vegan (vegetables, fruits, lentils)")
    user['DietType'] = int(input("Enter number: "))

    print("How often do you eat locally grown / local market food?")
    print("  0 = Often (most of the time)")
    print("  1 = Rarely (almost never)")
    print("  2 = Sometimes (occasionally)")
    user['LocalFoodFrequency'] = int(input("Enter number: "))

    # ── Transport ──
    print("\n--- 🚗 How do you travel? ---")
    print("What is your main way of getting around daily?")
    print("  0 = Bicycle")
    print("  1 = Car / bike (petrol/diesel vehicle)")
    print("  2 = Bus / auto / metro / rickshaw")
    print("  3 = I mostly walk")
    user['TransportationMode'] = int(input("Enter number: "))

    # ── Energy & Home ──
    print("\n--- ⚡ Energy & Home ---")
    print("What type of electricity do you use at home?")
    print("  0 = Mix (sometimes solar, sometimes regular grid)")
    print("  1 = Regular electricity from the grid (coal/gas based)")
    print("  2 = Solar panels or green energy provider")
    user['EnergySource'] = int(input("Enter number: "))

    print("Roughly how much electricity does your house use per month?")
    print("  (Check your electricity bill — look for 'Units' or 'kWh')")
    print("  Typical values: Small flat=100-200, Medium house=200-400, Large house=400+")
    user['MonthlyElectricityConsumption'] = float(input("Enter units/kWh (e.g. 250): "))

    print("What type of home do you live in?")
    print("  0 = Flat / Apartment")
    print("  1 = Independent house / Bungalow")
    user['HomeType'] = int(input("Enter number: "))

    print("How big is your home approximately?")
    print("  0 = Small  (1BHK or studio, under 600 sq ft)")
    print("  1 = Medium (2BHK, around 600-1200 sq ft)")
    print("  2 = Large  (3BHK or bigger, above 1200 sq ft)")
    size_input = int(input("Enter number: "))
    # Convert to approximate sq ft for the model
    size_map = {0: 500, 1: 900, 2: 1600}
    user['HomeSize'] = size_map.get(size_input, 900)

    # ── Shopping & Waste ──
    print("\n--- 🛍️ Shopping & Waste ---")
    print("How often do you use plastic bags, bottles, or packaging?")
    print("  0 = Often (daily use of plastic bags, plastic bottles etc.)")
    print("  1 = Rarely (I carry reusable bags, avoid plastic)")
    print("  2 = Sometimes (I try to avoid but not always)")
    user['UsingPlasticProducts'] = int(input("Enter number: "))

    print("How often do you buy new clothes?")
    print("  0 = Often (every month or more)")
    print("  1 = Rarely (only when really needed)")
    print("  2 = Sometimes (a few times a year)")
    user['ClothingFrequency'] = int(input("Enter number: "))

    print("Do you buy from eco-friendly or sustainable brands?")
    print("  (Brands that use natural materials, recycle packaging etc.)")
    print("  0 = No")
    print("  1 = Yes")
    user['SustainableBrands'] = int(input("Enter number: "))

    print("How do you usually throw away your waste / garbage?")
    print("  0 = Mix of different methods")
    print("  1 = Composting (wet waste like food scraps go to compost)")
    print("  2 = Everything goes in one bin (landfill / regular garbage)")
    print("  3 = I separate and recycle (paper, plastic, glass separately)")
    user['DisposalMethods'] = int(input("Enter number: "))

    # ── Awareness & Lifestyle ──
    print("\n--- 🌍 Awareness & Lifestyle ---")
    print("How aware are you about environmental issues?")
    print("  (Climate change, pollution, carbon footprint etc.)")
    print("  1 = Not aware at all")
    print("  2 = Slightly aware")
    print("  3 = Moderately aware")
    print("  4 = Quite aware")
    print("  5 = Very aware, I actively follow this topic")
    user['EnvironmentalAwareness'] = int(input("Enter number (1-5): "))

    print("Are you involved in any community or social activities?")
    print("  (Colony events, NGOs, cleanliness drives, local groups etc.)")
    print("  0 = Yes, very active")
    print("  1 = No, not really")
    print("  2 = Sometimes / a little")
    user['CommunityInvolvement'] = int(input("Enter number: "))

    print("How physically active are you in daily life?")
    print("  (Walking, sports, gym, cycling etc.)")
    print("  0 = Very active (daily exercise or physical work)")
    print("  1 = Not very active (mostly sitting, very little movement)")
    print("  2 = Moderately active (some days active, some not)")
    user['PhysicalActivities'] = int(input("Enter number: "))

    print("How much water does your household use per month?")
    print("  (Rough estimate is fine!)")
    print("  0 = Less  (under 2000 liters — small flat, 1-2 people)")
    print("  1 = Medium (2000-5000 liters — average family)")
    print("  2 = High  (above 5000 liters — large house or family)")
    water_input = int(input("Enter number: "))
    water_map = {0: 1500, 1: 3500, 2: 6000}
    user['MonthlyWaterConsumption'] = water_map.get(water_input, 3500)

    print("\n✅ Thank you! Calculating your sustainability score...\n")
    return user

In [8]:
def prepare_input(user_input, columns):
    full_input = {col: 0 for col in columns}
    
    for key in user_input:
        if key in full_input:
            full_input[key] = user_input[key]
    
    input_data = list(full_input.values())
    
    return input_data, full_input

In [9]:
user_partial = get_user_input()

input_data, full_user = prepare_input(user_partial, X.columns)

score = model.predict([input_data])[0]

print("Predicted Sustainability Score:", round(score, 2), "/5")

   🌱 SUSTAINABILITY LIFESTYLE QUESTIONNAIRE

--- 👤 About You ---


How old are you? (e.g. 20):  56


What is your gender?
  0 = Female   1 = Male   2 = Other / Prefer not to say


Enter number:  1


What type of area do you live in?
  0 = Village / Small town
  1 = Outskirts of a city (like Gandhinagar near Ahmedabad)
  2 = City center (like Ahmedabad, Mumbai, Delhi)


...

How physically active are you in daily life?
  (Walking, sports, gym, cycling etc.)
  0 = Very active (daily exercise or physical work)
  1 = Not very active (mostly sitting, very little movement)
  2 = Moderately active (some days active, some not)


Enter number:  1


How much water does your household use per month?
  (Rough estimate is fine!)
  0 = Less  (under 2000 liters — small flat, 1-2 people)
  1 = Medium (2000-5000 liters — average family)
  2 = High  (above 5000 liters — large house or family)


Enter number:  2



✅ Thank you! Calculating your sustainability score...

Predicted Sustainability Score: 1.67 /5


In [12]:
# ── CELL 13: Content Based Filtering Recommendation Engine ──
# How it works:
#   1. Find top 20 most similar users from dataset (cosine similarity)
#   2. From those, keep only users who scored HIGHER than current user
#   3. For each actionable feature, count how many better users
#      have a more sustainable habit than current user
#   4. Calculate percentage — "75% of similar high scorers do X"
#   5. For each bad habit found, give 3 levels of advice (Easy/Medium/Hard)
#   6. Acknowledge what user is already doing well

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# ── Sustainability ranking for each feature ──
# Higher rank number = more sustainable choice
# Used to check if similar user's habit is actually BETTER than current user's
sustainability_rank = {
    'TransportationMode'  : {3: 4, 0: 3, 2: 2, 1: 1},    # Walk > Bike > Public > Car
    'EnergySource'        : {2: 3, 0: 2, 1: 1},           # Renewable > Mixed > Non-Renewable
    'UsingPlasticProducts': {1: 3, 2: 2, 0: 1},           # Rarely > Sometimes > Often
    'DietType'            : {2: 3, 0: 2, 1: 1},           # PlantBased > Balanced > AnimalBased
    'DisposalMethods'     : {1: 4, 3: 3, 0: 2, 2: 1},     # Composting > Recycling > Combo > Landfill
    'ClothingFrequency'   : {1: 3, 2: 2, 0: 1},           # Rarely > Sometimes > Often
    'SustainableBrands'   : {1: 2, 0: 1},                 # Yes > No
    'CommunityInvolvement': {0: 3, 2: 2, 1: 1},           # High > Moderate > Low
    'LocalFoodFrequency'  : {0: 3, 2: 2, 1: 1},           # Often > Sometimes > Rarely
}

# ── Friendly labels so output reads in plain English ──
feature_labels = {
    'TransportationMode'  : {0: 'Bicycle', 1: 'Car/Petrol vehicle', 2: 'Public transport', 3: 'Walking'},
    'EnergySource'        : {0: 'Mixed energy', 1: 'Regular grid electricity', 2: 'Solar/Renewable energy'},
    'UsingPlasticProducts': {0: 'Uses plastic often', 1: 'Rarely uses plastic', 2: 'Sometimes uses plastic'},
    'DietType'            : {0: 'Mixed diet', 1: 'Mostly non-veg', 2: 'Mostly veg/plant-based'},
    'DisposalMethods'     : {0: 'Mixed disposal', 1: 'Composting', 2: 'Single bin/landfill', 3: 'Separate recycling'},
    'ClothingFrequency'   : {0: 'Buys clothes often', 1: 'Rarely buys clothes', 2: 'Buys clothes sometimes'},
    'SustainableBrands'   : {0: 'Does not buy eco brands', 1: 'Buys eco-friendly brands'},
    'CommunityInvolvement': {0: 'Very active in community', 1: 'Not involved', 2: 'Somewhat involved'},
    'LocalFoodFrequency'  : {0: 'Often eats local food', 1: 'Rarely eats local food', 2: 'Sometimes eats local food'},
}

# ── Tiered advice: Easy / Medium / Hard for each feature ──
# Each bad habit gets 3 levels so user can start from wherever they are comfortable
tiered_advice = {
    'TransportationMode': [
        "🟢 Easy   → For short distances under 1km, try walking instead of taking the vehicle",
        "🟡 Medium → Use bus, auto, or metro for your regular commute at least 3 days a week",
        "🔴 Hard   → Switch to cycling as your primary daily transport"
    ],
    'EnergySource': [
        "🟢 Easy   → Switch off appliances fully instead of leaving on standby — saves 10-15% electricity",
        "🟡 Medium → Ask your electricity provider if they offer a green/solar energy plan",
        "🔴 Hard   → Install a rooftop solar panel — many Indian states offer government subsidies for this"
    ],
    'UsingPlasticProducts': [
        "🟢 Easy   → Carry one reusable cloth bag whenever you go to buy groceries",
        "🟡 Medium → Replace plastic water bottles with a steel or copper bottle",
        "🔴 Hard   → Do a full audit of your kitchen and replace all single-use plastic containers"
    ],
    'DietType': [
        "🟢 Easy   → Have one fully vegetarian day per week (like a 'Meatless Monday')",
        "🟡 Medium → Replace one non-veg meal per day with a dal, rajma, or paneer based meal",
        "🔴 Hard   → Gradually shift to a mostly plant-based diet over 2-3 months"
    ],
    'DisposalMethods': [
        "🟢 Easy   → Keep two separate dustbins — one for wet waste (food) and one for dry waste",
        "🟡 Medium → Start composting your kitchen food scraps using a small compost bin",
        "🔴 Hard   → Set up full waste segregation at home — wet, dry, and hazardous separately"
    ],
    'ClothingFrequency': [
        "🟢 Easy   → Before buying new clothes, ask yourself if you really need them",
        "🟡 Medium → Try buying from thrift stores or second-hand apps like OLX for clothes",
        "🔴 Hard   → Commit to buying no new clothing for 3 months — use what you have"
    ],
    'SustainableBrands': [
        "🟢 Easy   → Next time you buy a product, spend 2 minutes checking if there is an eco-friendly version",
        "🟡 Medium → Replace 2-3 regular household products with eco-certified alternatives",
        "🔴 Hard   → Make sustainable brands your default choice for all future shopping"
    ],
    'CommunityInvolvement': [
        "🟢 Easy   → Share one environmental tip or article on your social media this week",
        "🟡 Medium → Join a local cleanliness drive or tree plantation event in your area",
        "🔴 Hard   → Regularly volunteer with an environmental NGO or start a small eco-initiative in your colony"
    ],
    'LocalFoodFrequency': [
        "🟢 Easy   → Visit your nearest local sabzi mandi or farmer market once this week",
        "🟡 Medium → Try to buy fruits and vegetables from local vendors instead of supermarkets",
        "🔴 Hard   → Source at least 70% of your food from local markets regularly"
    ],
}

# ── What user is already doing well — positive acknowledgements ──
positive_acknowledgements = {
    'TransportationMode'  : {3: "You already walk for most trips — that's fantastic! 🚶",
                              0: "You cycle regularly — great eco-friendly choice! 🚲",
                              2: "You use public transport — that's already a big win! 🚌"},
    'EnergySource'        : {2: "You use solar/renewable energy — amazing! ⚡🌞"},
    'UsingPlasticProducts': {1: "You rarely use plastic — keep it up! 🧴✅"},
    'DietType'            : {2: "Your mostly plant-based diet is already great for the planet! 🥗"},
    'DisposalMethods'     : {1: "Composting your waste is excellent — very few people do this! ♻️",
                              3: "You already recycle — that's a great habit! ♻️"},
    'ClothingFrequency'   : {1: "You rarely buy new clothes — that's very sustainable! 👗✅"},
    'SustainableBrands'   : {1: "You already buy from sustainable brands — well done! 🛒✅"},
    'CommunityInvolvement': {0: "You're very active in your community — that's inspiring! 🤝"},
    'LocalFoodFrequency'  : {0: "You often eat locally sourced food — great choice! 🌾"},
}


def get_content_based_recommendations(user_input_dict, user_score, df_original, X):
    """
    user_input_dict : dict of user's answers (from full_user)
    user_score      : the predicted score for this user
    df_original     : full dataframe with Rating column
    X               : feature dataframe without ParticipantID and Rating
    """

    # ── Step 1: Build user feature array ──
    user_array = np.array(
        [user_input_dict.get(col, 0) for col in X.columns]
    ).reshape(1, -1)

    # ── Step 2: Find top 20 most similar users using cosine similarity ──
    similarities = cosine_similarity(user_array, X.values)[0]
    similar_indices = np.argsort(similarities)[::-1][:20]

    # ── Step 3: Keep only similar users who scored HIGHER than current user ──
    better_users_indices = [
        idx for idx in similar_indices
        if df_original.iloc[idx]['Rating'] > user_score
    ]

    # If no better users found, use all top similar users
    if len(better_users_indices) == 0:
        better_users_indices = list(similar_indices[:10])

    total_better = len(better_users_indices)

    # ── Step 4: Count how many better users have a more sustainable habit ──
    # for each actionable feature
    actionable_features = list(sustainability_rank.keys())

    feature_improvement_count = {}   # how many better users have a better habit
    feature_best_value        = {}   # what is the most common better value

    for feature in actionable_features:
        user_val   = user_input_dict.get(feature, 0)
        user_rank  = sustainability_rank[feature].get(int(user_val), 0)

        better_count  = 0
        value_counts  = {}

        for idx in better_users_indices:
            their_val  = int(X.iloc[idx][feature])
            their_rank = sustainability_rank[feature].get(their_val, 0)

            # Only count if their habit is genuinely MORE sustainable
            if their_rank > user_rank:
                better_count += 1
                value_counts[their_val] = value_counts.get(their_val, 0) + 1

        if better_count > 0:
            feature_improvement_count[feature] = better_count
            # Most common better value among similar high scorers
            feature_best_value[feature] = max(value_counts, key=value_counts.get)

    # ── Step 5: Sort features by how many similar users suggest improvement ──
    # Feature with highest count = most agreed upon improvement
    sorted_improvements = sorted(
        feature_improvement_count.items(),
        key=lambda x: x[1],
        reverse=True
    )

    # ── Step 6: Build final recommendations ──
    recommendations = []
    positives       = []

    # Collect what user is already doing well
    for feature in actionable_features:
        user_val = int(user_input_dict.get(feature, 0))
        ack = positive_acknowledgements.get(feature, {}).get(user_val)
        if ack:
            positives.append(ack)

    # Build improvement recommendations
    for feature, count in sorted_improvements[:4]:   # top 4 improvements
        percentage   = round((count / total_better) * 100)
        user_val     = int(user_input_dict.get(feature, 0))
        best_val     = feature_best_value[feature]
        user_label   = feature_labels[feature].get(user_val,  str(user_val))
        better_label = feature_labels[feature].get(best_val, str(best_val))
        steps        = tiered_advice[feature]

        rec = (
             f"   Currently you : {user_label}\n"
             f"   They switched to: {better_label}\n"
             f"\n"
             f"   How to improve (pick your level):\n"
             f"   {steps[0]}\n"
             f"   {steps[1]}\n"
             f"   {steps[2]}"
            )
        recommendations.append(rec)

    return positives, recommendations

In [13]:
# ── CELL 14: Run and print results ──

positives, recommendations = get_content_based_recommendations(
    user_input_dict = full_user,
    user_score      = score,
    df_original     = df,
    X               = X
)

# Round score neatly
score_display = round(float(score), 2)

# Score tier
if score_display >= 4.5:
    tier = "🏆 Sustainability Champion"
elif score_display >= 3.5:
    tier = "🌿 Advanced"
elif score_display >= 2.5:
    tier = "🌱 Intermediate"
else:
    tier = "⚠️ Needs Improvement"

print("=" * 58)
print("        🌍 YOUR SUSTAINABILITY RESULTS")
print("=" * 58)
print(f"\n  Score : {score_display} / 5.0   |   Level : {tier}")

# Print what user is already doing well
if positives:
    print("\n✅ What you're already doing great:")
    for p in positives:
        print(f"   {p}")

# Print recommendations
if recommendations:
    print("\n📌 Areas to improve:")
    print("   (Based on people similar to you who scored higher)")
    print("-" * 58)
    print("\n  People similar to you who scored higher changed these habits:\n")
    for i, rec in enumerate(recommendations, 1):
        print(f"\n  {i}. {rec}")
        print()
else:
    print("\n🏆 Your lifestyle is already very close to top scorers!")
    print("   Consider inspiring others in your community!")

print("=" * 58)

        🌍 YOUR SUSTAINABILITY RESULTS

  Score : 1.67 / 5.0   |   Level : ⚠️ Needs Improvement

✅ What you're already doing great:
   You already recycle — that's a great habit! ♻️

📌 Areas to improve:
   (Based on people similar to you who scored higher)
----------------------------------------------------------

  People similar to you who scored higher changed these habits:


  1.    Currently you : Rarely eats local food
   They switched to: Sometimes eats local food

   How to improve (pick your level):
   🟢 Easy   → Visit your nearest local sabzi mandi or farmer market once this week
   🟡 Medium → Try to buy fruits and vegetables from local vendors instead of supermarkets
   🔴 Hard   → Source at least 70% of your food from local markets regularly


  2.    Currently you : Car/Petrol vehicle
   They switched to: Public transport

   How to improve (pick your level):
   🟢 Easy   → For short distances under 1km, try walking instead of taking the vehicle
   🟡 Medium → Use bus, auto, 